# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 💾 Charger les données

In [ ]:
import json
import ast
from pathlib import Path

def load_pyomo_data(input_path="../data/Regime_data.json"):
    """Charge les données JSON et convertit les clés string en types natifs."""
    input_file = Path(input_path)
    with open(input_file, "r") as f:
        data = json.load(f)

    def _convert_key(key):
        if not isinstance(key, str):
            return key
        if key.startswith("(") and key.endswith(")"):
            try:
                return ast.literal_eval(key)
            except Exception:
                return key
        try:
            return int(key)
        except Exception:
            return key

    # Convertir les dictionnaires de parametres indexes
    params = data.get("params", {})
    for pname, pval in list(params.items()):
        if isinstance(pval, dict):
            params[pname] = {_convert_key(k): v for k, v in pval.items()}

    cartesian = data.get("cartesian_data", {})
    for cname, cval in list(cartesian.items()):
        if isinstance(cval, dict):
            cartesian[cname] = {_convert_key(k): v for k, v in cval.items()}

    data["params"] = params
    data["cartesian_data"] = cartesian
    return data

# Charger les données
data = load_pyomo_data()

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.ALIMENTS = Set(initialize=data['sets']['ALIMENTS'])
model.INGREDIENTS = Set(initialize=data['sets']['INGREDIENTS'])
model.ARCS = Set(dimen=2, initialize=[(i,j) for i in model.ALIMENTS for j in model.INGREDIENTS])

## 🔹 Parameters

In [ ]:
model.Prix = Param(model.ALIMENTS, initialize=data['params']['Prix'], within=NonNegativeReals)
model.Calories = Param(model.ALIMENTS, initialize=data['params']['Calories'], within=NonNegativeReals)
model.DIETEJOUR = Param(model.INGREDIENTS, initialize=data['params']['DIETEJOUR'], within=NonNegativeReals)
model.QTEING = Param(model.ALIMENTS, model.INGREDIENTS, initialize=data['cartesian_data']['QTEING'], within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.ALIMENTS, domain=NonNegativeReals)

## 🔹 Constraints

In [ ]:
model.c0 = Constraint(expr=sum(model.Calories[a]*model.X[a] for a in model.ALIMENTS) >= 500)
model.c_for_0 = ConstraintList()
for i in model.INGREDIENTS:
    model.c_for_0.add(sum(model.QTEING[a,i] * model.X[a] for a in model.ALIMENTS) >= model.DIETEJOUR[i])

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.Prix[a] * model.X[a] for a in model.ALIMENTS), sense=minimize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')